# Cómo obtener propiedades termodinámicas de los diferentes paquetes ya existentes

- thermochem
    - iapws
    - janaf
    - burcat
- iapws
- coolprop

- thermochem.py
    - iapws
    iapws permite evaluar, solo para water:
        - h a partir de T y P
        - T de saturación a partir de P
        - P de saturación a partir de T
        - T a partir de h y P

Basado en International Association for the Properties of Water and
Steam. Lucerne, Switzerland. August 2007. 

Tiene los coeficientes de las funciones cargados en el propio script, no necesita acceder a otra base de datos.

In [22]:
import thermochem.iapws as iapws

w = iapws.Water() #genera el objeto agua

T = 373.15 # temperatura en K

P = 100000/100 # presión en Pa

h = w.h(P, T)  # entalpía en kJ/kg

print(f"Enthalpy {h} kJ/kg at {T} K and {P} Pa")

Enthalpy 2688.536040687925 kJ/kg at 373.15 K and 1000.0 Pa



No devuelve en qué estado de agregación está.

Si P < Psat, devuelve la entalpía de VSC

Si P >= Psat, devuelve la entalpía de LC o Lsat

Una aproximación a la h de Vsat se puede obtener evaluando en Psat-0.0001

In [ ]:
hv_sat = w.h(w.psat(T)-0.0001, T)
print(f"Enthalpy {hv_sat} kJ/kg at {T} K and {w.psat(T)-0.0001} Pa")

Enthalpy 2675.572017206459 kJ/kg at 373.15 K and 101417.97782131012 Pa


In [15]:
Psat = w.psat(T)  # presión de saturación en Pa

Tsat = w.Tsat(P)  # temperatura de saturación en K

print(f"Saturation pressure of water at {T} K: {Psat} Pa")
print(f"Saturation temperature of water at {P} Pa: {Tsat} K")

Saturation pressure of water at 373.15 K: 101417.97792131013 Pa
Saturation temperature of water at 10000.0 Pa: 318.9575482070235 K


In [28]:
Temp = w.T_ph(P/1000000, h)[1] #T(K) a partir de P (MPa) y h (kJ/kg). Para P < 4 MPa.

print(f"Temperature {Temp:.2f} K at {P/1000000} MPa and {h:.2f} kJ/kg")

Temperature 373.15 K at 0.001 MPa and 2688.54 kJ/kg


In [24]:
print(h)

2688.536040687925


In [ ]:
- thermochem.py
    - janaf

Accede a la base de datos del NIST-JANAF 

No sé si tendría mucho sentido comparar con este paquete 

In [29]:
import urllib.request
import sys
import pandas as pd
import thermochem.janaf as Janaf

# Configurar un opener global para urllib.request con un User-Agent personalizado

opener = urllib.request.build_opener()
opener.addheaders = [
    ("User-Agent", "Mozilla/5.0 (compatible; Python-thermochem-client/1.0)")
]
urllib.request.install_opener(opener)

importo urllib2.request


In [ ]:
db = Janaf.Janafdb()  # Crear el objeto de la base de datos

search_results = db.search("H2O") #Buscar resultados compatibles con H2O en la base de datos

print(search_results) #Estos serán los keys a utilizar para extraer los datos.

     formula                 name phase filename
534     CH2O         Formaldehyde     g    C-061
1213     H2O                Water     l    H-063
1214     H2O                Water     g    H-064
1215     H2O         Water, 1 Bar   l,g    H-065
1216     H2O        Water, 10 Bar   l,g    H-066
1217     H2O       Water, 100 Bar   l,g    H-067
1218     H2O       Water, 500 Bar    fl    H-068
1219     H2O      Water, 5000 Bar    fl    H-069
1220    H2O2    Hydrogen Peroxide     g    H-070
1221  H2O2Sr  Strontium Hydroxide    cr    H-071
1222  H2O2Sr  Strontium Hydroxide     l    H-072
1223  H2O2Sr  Strontium Hydroxide  cr,l    H-073
1224  H2O2Sr  Strontium Hydroxide     g    H-074
1225   H2O4S        Sulfuric Acid  cr,l    H-075
1226   H2O4S        Sulfuric Acid     g    H-076
1227   H2O4W        Tungstic Acid    cr    H-077
1228   H2O4W        Tungstic Acid     g    H-078


In [ ]:
#Extraer los datos, de acuerdo a la formula y fase elegida.

datos = db.getphasedata(formula="H2O", name="Water", phase="l", filename="H-063", cache=False)

print(datos.S([298, 300, 302])/18)        # Entropy in J/g/K

print("falta imprimir tabla")

Esta solicitando https://janaf.nist.gov/tables/H-063.txt
[3.8839371  3.912      3.93901667]


/home/caro/Documentos/Caro/Python_Cientifico/FinalProject/.venv/lib/python3.15/site-packages/thermochem/janaf.py:82: ParserWarning: Skipping line 12: Expected 8 fields in line 12, saw 9. Error could possibly be due to quotes being ignored when a multi-char delimiter is used.

  data = pd.read_csv(


- thermochem.py
    - burcat

This module extracts the information provided in *Third Millennium Ideal Gas
and Condensed Phase Thermochemical Database for Combustion with Updates from
Active Thermochemical Tables* by A. Burcat and B. Ruscic. 

It needs the actual database BURCAT_THR.xml to run, which is already included in the thermochem library.

